<!--
File: technical_memory.ipynb
Description: Running technical decision log for The Village. Records architectural decisions, tradeoffs, and gotchas over the project's lifetime.
Author: Jose-Jorge HERNANDEZ
Company: Parlee Conseiller, Inc.
Date: 2026-09-07
Last edit date: 2026-10-21
Version: 6.0.0
-->

# Technical Memory — The Village

This notebook is a running log of *why* — design decisions, tradeoffs weighed and rejected, gotchas hit and how they were resolved. It is not a changelog of *what* changed (`CHANGELOG.md`) and not a bug log (`docs/BUGS.md`, once one exists) — see each project's own docs for those.

## 2026-09-07 — Documentation overhaul & project history recovery

### Context
The repo's docs (`README.md`, `TODO.md`, `ARCHITECTURE.md`, `CHANGELOG.md`, `GDD.md`) had drifted out of sync with both each other and the actual codebase, and with a much longer project history that lived entirely outside the git repo (in school coursework going back to Feb 2024).

### Decision 1 — Resolving the TODO.md vs CLAUDE.md phase-completion conflict
`CLAUDE.md` claimed "all 6 phases done" (movement/combat, rooms, items, main menu, pause menu, real sprites) while `TODO.md` still showed most of those as unchecked, using an older, differently-numbered phase scheme (0–8) that was never updated after those milestones actually shipped.

**Resolution:** verified directly against the file tree (`scenes/ui/MainMenu.tscn`, `GameOver.tscn`, `HealthUI.tscn`, `scenes/world/rooms/Room1_Tutorial.tscn` + `Room2_Challenge.tscn`, `scenes/items/HeartPickup.tscn`, wired sprites `tileset_forest.png`/`slime.png`/`sword.png`) — all exist. `CLAUDE.md` was correct; `TODO.md` was stale. Rewrote `TODO.md` onto `CLAUDE.md`'s phase numbering so the two documents can't drift apart the same way again.

### Decision 2 — Reconstructing an accurate CHANGELOG from git history instead of the vague `[Unreleased]` block
The changelog only had one dated entry (`[0.0.1]` — 2026-05-25) with everything else bucketed under `[Unreleased]`. Real commit history was available and more precise, so it was used to rebuild dated release entries.

### Decision 3 — Extended lore preserved as a separate `LORE.md`, not merged into `GDD.md`
School research (see below) surfaced a much richer original narrative (protagonist Joe, antagonist Caver, four dimensions) that the current `GDD.md` explicitly scopes *out* of the prototype. Rather than re-inflate the lean prototype GDD, the fuller narrative was captured in a new root-level `LORE.md`, marked as dormant/future-vision material — preserves it for later without disturbing the working, intentionally-small current scope.

### Decision 4 — Asset migration from the school folder, with a licensing caveat
Copied into `assets/branding/`, `assets/reference/joe/`, and `assets/audio/`:
- Branding art (`thevillage*.png/.ico`) and Joe's character reference art/pixel frames — both the student's own coursework, low risk.
- Audio clips (`bg_music`, `button_click`, `forest`, `person-running-loop-245173`, `retro-jump-3-236683`, `pianochocolate_forest`) made for the earlier Unreal Engine 5 prototype.

**Gotcha to resolve before shipping:** the audio files' actual licenses were not verified during this pass — `person-running-loop-245173` and `retro-jump-3-236683` look like they're from a stock SFX site (ID-suffixed filenames, e.g. Pixabay/Freesound convention) and `pianochocolate_forest.mp3` is named after a real artist ("Pianochocolate") and may not be freely redistributable. None of these should be included in a public/shipped build until their license terms are confirmed.

**Deliberately NOT copied:** `800609_warxwell_16x16-pixel-chars.png` (a third-party pixel-character reference sheet found alongside Joe's art) — this reads as someone else's published asset pack used only as visual reference, not the student's own work, so it was left in the school folder rather than copied into the repo.

In [ ]:
%%bash
# Command used to reconstruct the dated CHANGELOG entries from real commit history
# (run from the repo root)
git log --all --date=short --stat --pretty=format:"=== %ad | %s ===" --reverse

### Project history summary (for future sessions)

The Village did **not** start as this Godot repo. Full lineage, recovered from `H:\OneDrive\Office\School\UVM`:

| When | Course | What happened |
|---|---|---|
| 2024-02 | Estructuras Narrativas para Videojuegos | Concept originates — first GDD draft, gothic medieval village, isometric action-RPG |
| 2024-06 | Diseño de Personajes | Protagonist Joe fully designed — character sheet + 8-frame pixel sprite |
| 2024-08 | Diseño de Escena y Componentes | Worldbuilding expands — level map, moodboards, HUD/item art |
| 2024-11 to 2024-12 | Motor de Videojuegos | Built as a working Unreal Engine 5 prototype across 9 graded assignments (movement, collision, rigging, level blockout, menu, sound, AI, save system) |
| *(~17-month gap, no documented activity)* | | |
| 2026-05-25 → present | — | Rebuilt from scratch in Godot 4.6, deliberately scoped down to a lean top-down prototype (no story, single enemy type, no saves) — this repo |

The current repo is a clean-slate reboot in a new engine, not a continuation of the UE5 codebase — no code or assets carried over directly (the UE5 project files were never located in the school folder, only its documentation, videos, and a handful of standalone art/audio assets).

## 2026-10-15 — Rebuild on the Chapel of Dreams architecture (Phases 0 and 1)

### Context
The Godot prototype (two rooms, slimes, placeholder Joe) had been dormant since September. Meanwhile game 2 of the trilogy, *Chapel of Dreams*, was built to beta with a much stronger method: generated art from one palette, manifests read at runtime, `PlayerTuning`-style constant files, and headless scripted test suites. The owner asked to build this game for real, "same mechanics, visuals" as *A Link to the Past*.

### Decisions (owner-confirmed where noted)
1. **Rebuild in place, not extend** (owner). The prototype's autoload web (EventBus/GameManager/SceneManager/InputHandler) did not fit Chapel's patterns, and its art was placeholder. Rebuilding on Chapel's base reuses ~1,000 lines of proven code (HitBox/HurtBox/Health/StateMachine/SpriteStrips/HitFlash) and its whole tooling. The prototype stays in git history (`cec47ca`).
2. **384×216, not ALttP's 256×224** (owner). Consistency across the trilogy won over authenticity. Consequence: a screen is 24×13.5 tiles, so Phase 3's screen-scrolling areas will be 24×14 tiles with half a row hidden behind the HUD band, or the HUD overlays the field — to decide in Phase 3.
3. **Light/Dark mirror worlds** (owner). The four lore dimensions become Dark World regions; `palette.py` already carries a `DW_*` set so the Dark World can be the same generators with a palette swap.
4. **Feet-origin actors with a feet-only collision box** (10×6). This is how ALttP lets Link's head overlap the tile above a wall, and it makes y-sorting trivial (sort by origin).
5. **48×48 frames for Joe.** The sword sweeps around him; drawing it into the frame (instead of a separate rotated sprite) keeps the art exact and lets the manifest carry the hit window.
6. **Runtime TileSet from a JSON manifest** instead of a hand-maintained `.tres`: the generator is the single source of truth for which tiles are solid.
7. **Corner nudge via `test_move`**: when a single-axis step is fully blocked, probe 1..6 px sideways; the first clear offset slides Joe that way at walk speed. Diagonal input never nudges (the slide already handles it).

### Gotchas met
- Case-insensitive filesystem: `Tools/` vs `tools/` produced a *file* named `tools` (see `docs/BUGS.md`).
- `Input.action_press` is seen on the next physics frame: tests wait 2 frames.
- Naming a Node method `_set` would override `Object._set`.


In [ ]:
# The corner-nudge core (src/player/Player.gd, v1.0.0), kept here because its tuning is a feel decision.
func _try_corner_nudge(step: Vector2, delta: float) -> void:
	var perp := Vector2(step.y, step.x).abs()
	for k in range(1, PlayerTuning.CORNER_NUDGE_PX + 1):
		for side in [-1.0, 1.0]:
			var off: Vector2 = perp * side * float(k)
			if not test_move(global_transform.translated(off), step):
				move_and_collide(perp * side * PlayerTuning.NUDGE_SPEED * delta)
				return


## 2026-10-15 — Phase 2: combat core

### Decisions
1. **Shield as a passive cone, not a button.** ALttP's shield blocks automatically while Link faces the projectile. Joe's `_on_hurt` asks `HitBox.blockable` and `blocks_from(source)` (dot product with the facing >= 0.6, i.e. a ~106 degree cone) and only in `idle`/`walk`/`charge`. Swinging, spinning, lifting or carrying leaves him open — the same trade-off ALttP makes.
2. **Projectiles tell the target, the target tells the projectile.** `HurtBox` emits `hurt` *before* `HitBox.landed`, so Joe can call `deflect()` on the rock first and the rock's own `landed` handler then sees `deflected` and stays alive to bounce away. No extra signal needed.
3. **Hit-stop is per actor (`freeze`), not `Engine.time_scale`.** Kept from Chapel: it stays deterministic in tests and does not stall the camera shake, HUD or other actors. Joe pauses his `StateMachine` node's physics process during a freeze.
4. **Thrown objects move on the ground, only the sprite flies.** The body and HitBox travel at ground level (walls and enemies are hit where they stand) while the sprite follows a falling parabola from carry height — ALttP's model, and it keeps collision honest.
5. **Spin as one shared strip with per-facing start frames.** Frame *i* holds the sword at i×45°; the manifest maps each facing to its first frame. One 8-frame strip instead of four, and `flip_h` is turned off for the spin because the left-facing frames are already drawn mirrored.
6. **Enemies overlap Joe and each other.** Like ALttP, enemy bodies collide only with the world and props; damage is by touch. It avoids enemies body-blocking Joe in corridors.
7. **A minimal HUD now (hearts + gems) instead of in Phase 4.** Without it, damage and drops were invisible to a playtester; the full HUD remains Phase 4.

### Gotchas
- In `-s` test scripts the autoloads are reached with `root.get_node(...)`; a static reference compiles before they exist (Chapel lesson, applied up front, no failure hit).
- A windowed capture with live guards produced shots of Joe blinking under a guard: captures now switch his hurt box off.


In [ ]:
# Shield check (src/player/Player.gd v2.0.0)
func blocks_from(source_position: Vector2) -> bool:
	if not fsm.current_state in PlayerTuning.SHIELD_STATES:
		return false
	var to_src := source_position - (global_position + PlayerTuning.BODY_CENTER)
	if to_src.length() < 0.01:
		return false
	return to_src.normalized().dot(Vector2(facing)) >= PlayerTuning.SHIELD_DOT


## 2026-10-15 — Phase 3: overworld engine

### Decisions
1. **One global coordinate system for the overworld.** Each area JSON carries `origin_tiles`; neighbours share edges. Crossing an edge needs no coordinate translation: Joe keeps walking, the World notices his feet left the current rect and asks `overworld_at(feet)` which area owns that point. This also made the lint simple (a global tile -> area map).
2. **Only one area alive at a time** (two during the 0.6 s slide). Cheap, and it gives ALttP's behaviour for free: enemies and bushes reset when you come back.
3. **The camera belongs to the World, not to Joe.** It is clamped per area, centred in rooms smaller than the view, and tweened during scrolls. Test arenas keep the simple child camera.
4. **Ledges are solid tiles plus a flag.** Rather than one-way collision (which Godot defines along a body's local up, awkward top-down), a ledge cell is solid and carries a `ledge` custom-data flag; walking south into it switches Joe to a `hop` state that flies him over the ledge cells without collision. The lint's reachability search understands the one-way edge.
5. **Doors are walkable cells in the house's footprint**, triggered by Joe's feet entering the cell while facing up; exit mats trigger when facing down. No Area2D triggers to keep in sync with the tiles.
6. **Generator owns pairing.** `gen_areas.py` links each house door with its interior's mat in both directions, so a door can never lead to a room without a way back (and the lint re-checks it).
7. **Screen = 24x14 tiles.** 216 px is 13.5 tiles; a 14-tile screen lets the camera scroll 8 px inside one screen rather than leaving half-tiles at area edges.

### Gotchas
- A new `class_name` made the headless suite hang (not fail) until `--import` ran; all Godot runs are now wrapped in `timeout`.
- Nine-slice autotiling must treat off-area cells as the same terrain, or every road shows a grass lip at area seams.
- My first capture script released the stick one frame before Joe reached the edge - a reminder that a windowed shot that "shows nothing happened" needs the timing checked before the code is blamed.


In [ ]:
# How the World finds where Joe went when he leaves an area (src/world/World.gd v1.0.0)
func overworld_at(p: Vector2) -> String:
	for aid in areas:
		if areas[aid]["kind"] == "overworld" and rect_of(aid).has_point(p):
			return aid
	return ""


## 2026-10-15 — Phase 4: HUD, menus, saves, dialogue

**Pausing model.** Every modal screen (dialogue, pause menu, item menu, Game Over) sets `get_tree().paused` and
lives on a CanvasLayer with `PROCESS_MODE_ALWAYS`. The alternative — a "busy" flag every actor checks — was rejected:
enemies, pickups' blink timers, Juice hit-stops and the HUD caption tween would each need to know about it. With a
real pause the world simply stops; only `Audio` (already `ALWAYS`) and `MenuList` (sets itself `ALWAYS`) needed care.

**Talk before lift, and the 0.15 s cooldown.** Interact both talks and lifts (ALttP's A button). Talking wins
because a pot next to a villager must not block the conversation. The box closes on the *same* button that opened
it; the box consumes the event in `_unhandled_input` and unpauses, but Joe's state machine polls
`Input.is_action_just_pressed("interact")` in the same frame and would immediately re-open the talk. Freezing Joe
for `TALK_COOLDOWN` after `closed` makes the polled press expire first (a test checks it).

**Talk points instead of physics queries.** Villagers and signs answer a tiny contract (`talk_point`, `talk`, `rest`)
and Joe measures distance from a probe one tile ahead — the same pattern as `liftable`. Boni behind her counter
just moves her talk point one tile forward (`across`), so the counter needs no special "talk-through" tile.

**Saving at the area's entry point.** The save stores where Joe *entered* the current area, not where he stands.
A loaded game therefore never starts inside a wall, mid-ledge-hop or next to an enemy, and Game Over's CONTINUE
uses the same point. Facing at entry is stored too so Joe looks into the room he walked into.

**Dialogue as data with first-meeting flags.** `dialogue.json` entries have `pages`, and optionally `once` + `flag`:
the first talk shows `once` and sets the flag in `GameState.flags`, which is saved — so story progress and "have I
met Tato" are the same mechanism. The area generator's lint loads the same file, so a typo in a sign key fails the
build instead of showing an empty box.

**Per-effect SFX level targets.** The old self-check accepted only peaks between -6.1 and -2.9 dB, which would have
forced the menu tick and text blip to be as loud as a sword hit. Each effect now has its own target in the `SFX`
table and the check is `|peak - target| <= 0.6 dB`.

**Gotchas paid for this phase** (also in `docs/BUGS.md`): `round()` returns a Variant (inference warning = error
here; use `roundf`); a `-s` test that names an autoload-using class statically fails to compile and silently
half-builds the World (read constants through the instance); long inline heredocs break the shell tool, so patches
were written as script files.

The cell below cross-checks the generated areas against the dialogue file and the villager sprite manifest — the
same rule the `gen_areas.py` lint enforces — and prints which text each villager and sign uses.

In [ ]:
import json, os

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'technical_memory' else os.getcwd()
areas_dir = os.path.join(ROOT, 'src', 'world', 'areas')
dialogue = json.load(open(os.path.join(ROOT, 'assets', 'data', 'dialogue.json'), encoding='utf-8'))
sprites = set(json.load(open(os.path.join(ROOT, 'assets', 'sprites', 'npcs', 'npcs.json'), encoding='utf-8'))['npcs'])
index = json.load(open(os.path.join(areas_dir, 'index.json'), encoding='utf-8'))

problems = []
for aid in index['areas']:
    a = json.load(open(os.path.join(areas_dir, aid + '.json'), encoding='utf-8'))
    for n in a.get('npcs', []):
        ok = n['id'] in dialogue and n['id'] in sprites
        print(f"{aid:12s} villager {n['id']:15s} at {n['cell']} -> {dialogue.get(n['id'], {}).get('name', '?')}")
        if not ok:
            problems.append((aid, n['id']))
    for s in a.get('signs', []):
        print(f"{aid:12s} sign     {s['text']:15s} at {s['cell']}")
        if s['text'] not in dialogue:
            problems.append((aid, s['text']))
print('problems:', problems or 'none')

## 2026-10-15 — Phase 5: Items I

**One item-get path.** Chests, villager gifts, loose heart pieces and shop purchases all end in
`World.item_get(kind)`: `grant()` changes the state, Joe `hold_up()`s the icon, the fanfare plays and the kind's
text is shown; the dialogue's `_after_dialogue` callable lowers the item. Having one path is what made Janga's gift
cheap: `DialogueData.conversation()` returns `{pages, gift}` and the World chains `item_get` after the lines.

**Interactions dispatched by class, not new contracts.** Chests, key blocks and wares joined the existing
`talkable` group (`talk_point`/`talk`/`rest`) so Joe's probe finds them for free; the World branches on
`is Chest` / `is KeyBlock` / `is ShopItem` before plain talk. A separate "interactable" contract was considered and
rejected: it would have duplicated the probe and the priority rules (talk before lift) for no gain.

**Stun as a HitBox element.** The boomerang reuses the combat contract with `damage = 0, element = "stun"`;
`BaseEnemy._on_hurt` checks the element first. Enemies need no new detection code, and later elements (fire rod,
ice rod) slot into the same switch.

**Bombs hurt Joe through the normal pipeline.** A bomb carries two HitBoxes - one on PlayerHitBox (enemies,
bushes) and one on EnemyHitBox (Joe, not blockable) - instead of special-casing Joe. Cracked rocks are a group
(`bombable`) with `blast()`, so later bombable walls and the hammer can reuse it.

**World persistence as flags.** Every one-time thing in the world is a GameState flag named
`<kind>:<area>:<x>,<y>` (wares: `ware:<id>`), checked when an area is built. Flags were already saved, so chests,
rocks, locks and heart pieces survive area changes and saves with no new save fields. Torches deliberately do NOT
persist (ALttP rooms reset); a chest they revealed is remembered once opened.

**Lint knows about opening things.** Reachability treats key blocks and cracked rocks as open, and the lint
separately requires keys >= key blocks and torches for every hidden chest, so a sealed area cannot ship.

**Gotchas:** `hidden` is a CanvasItem signal (renamed the chest flag `concealed`); `filter()` returns an untyped
Array (type it, or the headless suite hangs on the parse error); `HurtBox.recheck()` must not query overlaps while
monitoring is off; tests must re-face Joe after an item-get (he faces down) and wait out the 0.2 s use pose.

The cell below lists every one-time object in the generated world with the flag that remembers it - the same names
the World and the saves use.

In [ ]:
import json, os

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'technical_memory' else os.getcwd()
areas_dir = os.path.join(ROOT, 'src', 'world', 'areas')
index = json.load(open(os.path.join(areas_dir, 'index.json'), encoding='utf-8'))
keys = locks = 0
for aid in index['areas']:
    a = json.load(open(os.path.join(areas_dir, aid + '.json'), encoding='utf-8'))
    for o in a.get('objects', []):
        x, y = o['cell']
        extra = ' -> ' + o['item'] + (' (hidden)' if o.get('hidden') else '') if o['kind'] == 'chest' else ''
        print(f"{o['kind']}:{aid}:{x},{y}{extra}")
        keys += o.get('item') == 'small_key'
        locks += o['kind'] == 'key_block'
    for w in a.get('wares', []):
        print(f"ware:{w['ware']}  ({aid} counter at {w['cell']})")
print('small keys:', keys, ' key blocks:', locks)

## 2026-10-16 — Phase 6: dungeon framework + the Moss Shrine

**Dungeons reuse the overworld engine.** A dungeon is just more areas (`kind: "dungeon"`) on their own room grid,
grouped by a `dungeon` id. `group_of()` / `area_at()` replaced the overworld-only lookup, so ALttP room-to-room
scrolling came for free. The only dungeon-specific scroll tweak is the push: walls are two tiles thick, so Joe
steps 40 px in (14 px would leave him standing inside the doorway - and inside a shutter that closes behind him).

**Doors live in both rooms.** Only the current room exists in the tree, so a door drawn only on one side would let
Joe scroll into the other room straight through a locked door. Every shared door is listed in both rooms with one
id; its state comes from flags (`door:<id>`, or the shutter's `when` flag), so both copies always agree. The boss
door is the exception: it is open when the room is built and shuts once Joe is inside (after the scroll), which
needs no mirror.

**Room state as flags, not live checks only.** A cleared room sets `cleared:<room>` once; its shutter then stays open
even though enemies respawn when the room is rebuilt (otherwise coming back from the room behind the shutter
would trap Joe). Switches are permanent for the same reason.

**Pits on their own physics layer.** Joe must walk into pits (he falls), enemies must not. A second TileSet physics
layer (layer 9) puts a wall polygon on pit cells that only enemies and push blocks collide with; Joe checks the
`pit` custom-data flag under his feet instead.

**Keys per dungeon.** ALttP keys belong to their dungeon. `GameState.keys` became a property over `key_pools`
keyed by the dungeon id ("" = overworld), switched by the World on every area change, so every existing caller
(`add_keys`, key blocks, the HUD) works unchanged.

**Lint that understands progression.** Reachability treats doors, blocks, cracked rocks and heavy rocks as openable,
and separate rules check what opens them: small keys per dungeon >= distinct locked doors, a big key wherever there
is a big door or chest, a switch for every switch-revealed chest, a glove somewhere for heavy rocks.

**Gotchas:** a variable named like a method in the same class breaks the whole class (`reveal`); an Area2D does not
reliably report a StaticBody2D moved by a tween (the block now presses the switch it stops on directly); a reward
spawned under Joe is collected instantly (the seal appears higher up); a rebuilt room frees its nodes, so tests must
fetch doors/switches again.

The cell below walks the shrine's progression from the generated data: which room needs which key or flag.

In [ ]:
import json, os

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'technical_memory' else os.getcwd()
areas_dir = os.path.join(ROOT, 'src', 'world', 'areas')
index = json.load(open(os.path.join(areas_dir, 'index.json'), encoding='utf-8'))
print('dungeons:', index.get('dungeons'))
for aid in index['areas']:
    a = json.load(open(os.path.join(areas_dir, aid + '.json'), encoding='utf-8'))
    if a['kind'] != 'dungeon':
        continue
    doors = ['%s %s(%s)' % (o['dir'], o['type'], o.get('when') or o['id']) for o in a['objects'] if o['kind'] == 'door']
    chests = ['%s%s' % (o['item'], ' [' + o['reveal'] + ']' if o.get('reveal') else '') for o in a['objects']
              if o['kind'] in ('chest', 'big_chest')]
    print('%-13s grid %s  doors: %s  chests: %s  %s' % (aid, a['grid'], ', '.join(doors) or '-', ', '.join(chests) or '-',
          a.get('clear_flag', '') or a.get('boss_flag', '')))

## 2026-10-19 — Phase 7: the rest of Act 1

**Dungeons as data plus one helper.** The Moss Shrine was hand-coded as `d1_rooms()`. Rather than copy its inner
closure three times, `make_room(rooms, id, gx, gy, doors, fill, extra)` became the shared builder and `DUNGEONS`
lists every dungeon with its table entry; each gets its own room grid `DUNGEON_STRIDE` tiles apart. Ram's Hollow,
the Frost Vault and the keep are ~25 lines each.

**Boss weaknesses through one hook.** ALttP bosses each need a trick or a weapon. `BaseEnemy._resists(hit)` runs
before stun and damage and makes a resisted hit clank; the three bosses override it with one line each (dazed,
`element == "fire"`, `element == "orb"`). Elements already existed on HitBox (the boomerang's "stun"), so fireballs
and Caver's batted orbs only had to set theirs. No new detection code anywhere.

**The batted orb is a two-box object.** It carries a HitBox (EnemyHitBox layer, hurts Joe) and a HurtBox that
listens to Joe's attacks; when struck it moves its HitBox to the PlayerHitBox layer with element "orb" and homes on
its caster. The shade resists everything else, so the only way to win is the Agahnim rally - with zero special
cases in Joe's code.

**The act ending is a dungeon property.** The keep's table entry has `ending: "act1"` and `seal: -1`; the generic
boss-defeated path checks it and calls `play_ending` (last words -> `EndCard` -> save outside the keep -> title)
instead of spawning a reward. Phase 8 can pick up the Act 2 opening from that save and the `act1:done` flag.

**Generic first, tweak by content.** The seal gate is a "glyphless" object: it sits on plain floor cells (so the
doorway lint and door warps work unchanged) and covers them with a 2x2 solid body until the flag opens it.

**Gotchas:** test scripts must not name autoload-dependent classes statically - that now includes the bosses and
even `AreaBuilder` (its dependency Bush uses Audio); a 32-px talkable needs its talk point where Joe's 1-tile probe
reaches from either half; a room rebuild relights nothing (torches go out), which tests must respect.

The cell below lists every dungeon with its rooms, treasure and boss, straight from the generated data.

In [ ]:
import json, os

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'technical_memory' else os.getcwd()
areas_dir = os.path.join(ROOT, 'src', 'world', 'areas')
index = json.load(open(os.path.join(areas_dir, 'index.json'), encoding='utf-8'))
BOSSES = {'Y': 'Grand Slime', 'U': 'Iron Ram', 'F': 'Frost Wraith', 'V': "Caver's shade"}
for did, info in index['dungeons'].items():
    rooms, treasure, boss = [], [], '-'
    for aid in index['areas']:
        a = json.load(open(os.path.join(areas_dir, aid + '.json'), encoding='utf-8'))
        if a.get('dungeon') != did:
            continue
        rooms.append(aid)
        treasure += [o['item'] for o in a['objects'] if o['kind'] == 'big_chest']
        for g, name in BOSSES.items():
            if any(g in r for r in a['rows']):
                boss = name
    print('%-5s %-14s seal %2s  rooms %d  big chest %-10s boss %-14s %s' % (
        did, info['name'], info['seal'], len(rooms), ','.join(treasure) or '-', boss, info.get('ending', '')))

## 2026-10-20 — Phase 8: the Dark World and the Moon Mirror

**Same coordinates, different group.** ALttP's mirror sends Link to "the same spot" in the other world. Instead of a
mapping table, every Dark World twin sits at exactly the same `origin_tiles` as its Light World area and belongs to
a second scroll group (`overworld_dark`). `area_at(point, group)` already existed for scrolling, so a world warp is
just "rebuild the area of the other group under Joe's feet" - one function (`_world_warp`) serves the mirror, its
return shimmer and the dream rifts.

**Generated twins, not hand-made maps.** `dark_twin()` derives each Dark World area from its Light World rows:
strip villagers, signs and one-time objects, apply a small per-area edit (the dry pond), add enemies with
`place_near` (nearest free cell, deterministic), then the usual NPC/sign/object tables. The twins inherit every
Light World layout change for free, which is exactly the "two worlds, one map" pillar.

**Recoloured atlas, not re-drawn.** `gen_dark.py` maps the Light World's signature colours to the `DW_*` palette
explicitly and pulls every other colour toward purple, desaturated and darker. The manifest is shared, so the dark
TileSet is the same cells with another texture (`AreaBuilder.tileset(dark)`).

**Mirror rules.** The mirror works only in the Dark World. After the warp the tile under Joe is checked
(`get_collision_polygons_count`); if solid he is thrown back. Otherwise a portal `Rift` arms only after he steps
off it (he arrives standing on it), like ALttP's sparkle.

**Lint for cross-world reachability.** A Light World object marked `via_mirror` is exempt from Light World
reachability but its Dark World twin cell must be reachable - mutation-tested by un-drying the pond.

**Gotchas:** z_index -1 hides flat props under the TileMapLayer (the Moss Shrine's switch had been invisible since
Phase 6 - caught only by a screenshot); `World.grant()` needs a branch per new kind.

The cell below compares each Light World area with its Dark World twin: name and cells that differ.

In [ ]:
import json, os

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'technical_memory' else os.getcwd()
areas_dir = os.path.join(ROOT, 'src', 'world', 'areas')
index = json.load(open(os.path.join(areas_dir, 'index.json'), encoding='utf-8'))
load = lambda aid: json.load(open(os.path.join(areas_dir, aid + '.json'), encoding='utf-8'))
for aid in index['areas']:
    a = load(aid)
    if a['kind'] != 'overworld' or a.get('world') == 'dark':
        continue
    d = load('dw_' + aid)
    diff = sum(1 for r1, r2 in zip(a['rows'], d['rows']) for c1, c2 in zip(r1, r2) if c1 != c2)
    print('%-13s -> %-16s same origin %s, %3d cells differ' % (a['name'], d['name'], a['origin_tiles'] == d['origin_tiles'], diff))
print('Act 2 spawn:', index['act2_spawn'])

## 2026-10-21 — Phase 9: Umbra (grapple, dark rooms, crystals)

**The grapple is a body-less projectile plus a player state, not physics.** The claw is a plain `Node2D` that steps
along Joe's facing and asks two questions each frame: is a `hookable` within `LATCH_RADIUS`, and is the next ground
point a wall? "Wall" means a solid tile *not* flagged `low`, or any body on the world/props layers. Water and pits
got the new TileSet flag `low` instead of a new physics layer: water must stay solid to Joe, so the grapple cannot
use the collision layers to tell "solid but crossable" from "solid". On a bite, `Player.grapple_pull()` switches
to state `pulled`, which clears Joe's collision mask and slides him to the post. Turning collision off was the
simplest way over pits and water; the pit check is not run in that state, so he does not fall mid-flight.

**Where Joe lands is a level-design contract, enforced by the lint.** The first draft stopped Joe 14 px short of
the post, which leaves his feet on the wrong side of a cell boundary. Coming from the right, that is over the pit.
The pull now targets exactly one tile before the post at a cell's feet height. `gen_areas.py` mirrors this in
`grapple_hops`: from any reachable cell, a straight line over open ground, pits or water to a post at most seven
tiles away reaches the cell in front of it, and only if that cell is walkable. With the hop disabled, the lint
reports six unreachable things (the island's heart, the east room's switch, compass and lurker). That proves the
reachability check really depends on the grapple.

**Dark rooms are one sprite, not lighting.** A Godot `PointLight2D` + `CanvasModulate` would need every tile and
sprite to carry lighting-friendly materials and costs a lot for a pixel-art look. Instead `gen_umbra.py` writes two
768x432 masks (twice the view, so Joe can stand anywhere on screen): opaque night with a soft, stepped hole of 30 or
78 px. The World draws one at z 100 centred on Joe. The HUD and boxes are CanvasLayers, so they stay visible. "All
torches lit" hides it, reusing the torch signal already used for hidden chests.

**Crystal dungeons have no `seal` key at all.** `_spawn_boss_reward` treats `seal < 0` as "no reward" (the keep). A
missing key defaults to 0, so a crystal dungeon must *not* carry `seal: -1`; it carries `crystal` and the branch
checks `info.has("crystal")` first. The lint reads seals with `.get("seal", -1)`.

**Dark World-only lands.** The vale has no Light World twin, so `area_at(p, "overworld")` returns "" there. The
mirror says `mirror_void` instead of silently doing nothing. The twin test in `test_dark.gd` skips such areas.

**Gotchas:** JSON numbers load as floats (`[0.0, -14.0] != [0, -14]`): compare positions as Vector2. A dialogue
box types its text, so tests wait ~2 s before reading `visible_text()`. `EnemyShot` draws one plain texture, so
shot art must be a single frame.

The cell below lists every hook post and the lint's grapple links from the generated data:


In [ ]:
import json, os, sys

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'technical_memory' else os.getcwd()
sys.path.insert(0, os.path.join(ROOT, 'tools'))
import gen_areas as G

areas_dir = os.path.join(ROOT, 'src', 'world', 'areas')
for aid in ('dw_umbra_vale', 'd4_east'):
    a = json.load(open(os.path.join(areas_dir, aid + '.json'), encoding='utf-8'))
    rows = a['rows']
    solid = G.solid_map(rows)
    posts = [tuple(o['cell']) for o in a['objects'] if o['kind'] == 'hook_post']
    links = sorted({(x, y, hop) for y in range(len(rows)) for x in range(len(rows[0])) if not solid[y][x]
                    for hop in G.grapple_hops(rows, solid, x, y)})
    print(aid, 'hook posts:', posts, '- grapple links:', len(links))
    print('  e.g.', links[:3])


## 2026-10-22 — Phase 10: Ignis (lava, the ice rod)

**Lava is a pit with two extra flags, not a new system.** Joe already falls into anything flagged `pit`; enemies
already treat the pit physics layer as a wall; the grapple already flies over `low` cells. So a lava cell is
`pit + low + lava`, and the only new rule is the ice rod's. Burning costs what a fall costs. The sound is chosen
by checking the `lava` flag under Joe as he falls (`sizzle` instead of `fall`).

**Freezing edits the TileMapLayer in place.** `AreaBuilder.freeze_at` swaps the lava piece for the `crust` cell
(no flags, no collision) and remembers which lava piece was under it. A `SceneTree` timer puts it back. Shooting the
same crust again must extend it, so every freeze bumps a per-cell *generation* and a timer only thaws when its
generation is still the newest. Real-time deadlines (`Time.get_ticks_msec`) were rejected: under
`--fixed-fps 60` the game clock runs faster than the wall clock, and the tests would disagree with play. Rebuilding
an area frees its TileMapLayer, so the timer checks `is_instance_valid` first. The timer is created with
`process_always = false`, so crust does not melt while a menu or a dialogue box has paused the tree.

**The lint walks over ice the way the bolt flies.** `ice_hops`: from a reachable cell, an unbroken run of lava
straight ahead leads to the first walkable cell past it, within `ICE_REACH` (8) tiles. Combined with
`grapple_hops`, this makes the order of the Dark World land explicit. The barrens' lava river has hook posts,
not a short crossing, so the grapple (Umbra) is required before Ignis.

**Shared random streams are fragile.** `gen_tiles.py` draws the whole atlas from one `random.Random`. Inserting
the lava cells before the shrine would have changed the shrine's pixels, so Ignis uses its own stream. A cell-by-cell
diff against the previous atlas confirmed nothing old changed.

**Bosses that come and go hide their boxes.** The Magma Wyrm is the first enemy that is untouchable for part of
its cycle: under the lava it turns off its sprite, hurt box (deferred `monitoring`) and contact box. It looks for
its lava cells lazily on its first rise, because enemies enter the tree before the room's TileMapLayer does.

The cell below counts the lava cells and the lint's ice crossings in the generated data:


In [ ]:
import json, os, sys

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'technical_memory' else os.getcwd()
sys.path.insert(0, os.path.join(ROOT, 'tools'))
import gen_areas as G

areas_dir = os.path.join(ROOT, 'src', 'world', 'areas')
for aid in ('dw_cinder_barrens', 'd5_east', 'd5_boss'):
    a = json.load(open(os.path.join(areas_dir, aid + '.json'), encoding='utf-8'))
    rows = a['rows']
    solid = G.solid_map(rows)
    lava = sum(r.count('!') for r in rows)
    hops = {(x, y, hop) for y in range(len(rows)) for x in range(len(rows[0])) if not solid[y][x]
            for hop in G.ice_hops(rows, solid, x, y)}
    print(aid, 'lava cells:', lava, '- ice crossings:', len(hops))


## 2026-10-23 — Phase 11: Tempus (tick blocks, the hammer, pegs)

**Tick blocks are actors, not tiles.** A TileMapLayer cell could have been swapped like the lava crust, but a
block must *not* rise under Joe, and that needs a physics query at the moment of rising and a "pending" state
afterwards. That is per-block logic, so each block is a `StaticBody2D` made by AreaBuilder from its glyph (like
bushes and pots). Being glyph actors also means they, and the pegs, reset whenever an area is rebuilt. That is
ALttP's behaviour for pegs, and the reason they are not one-time OBJECTS with GameState flags.

**One clock, owned by the World.** Blocks do not time themselves: `World._run_clock` counts down while the
current area has blocks and flips `tick_phase`, so every block of an area always agrees and a scroll or a paused
tree (dialogue, menus) stops the clock naturally. The phase is kept across areas, so a block's state on entry
depends on time. That is accepted (listed in KNOWN_ISSUES) rather than resetting the phase per area.

**Floor props drawn under actors, again.** Phase 8 learned that a negative z_index hides a prop under the ground.
A sunk tick block or a flat peg still has to be drawn *under* Joe when he walks over it, and the y-sort orders by
node position. So when they go down, the node moves up a few pixels (`SUNK_LIFT` / `FLAT_LIFT`) while the sprite
moves down by the same amount: it looks the same, but sorts earlier.

**A lint blind spot, documented instead of hacked around.** The reachability check seeds each overworld area from
its entries, and a dungeon's exit spawn counts as an entry. The clocktower's exit is on the far side of the
molten time, so the lint can never show that the ice rod gates Tempus. The gate is real in play (a dungeon can
only be entered from that side), so `test_tempus.gd` proves it in the game instead of the lint growing a special
case.

**Gotcha:** a fixed frame wait after a scroll is not enough when Joe starts a few tiles from the edge. The
scroll's end writes Joe's position and entry point after the next test has already moved him. Tests now wait for
`World.transitioning` to clear.

The cell below shows the clock's alternation on the generated wastes fence:


In [ ]:
import json, os

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'technical_memory' else os.getcwd()
a = json.load(open(os.path.join(ROOT, 'src', 'world', 'areas', 'dw_clockwork_wastes.json'), encoding='utf-8'))
fence = a['rows'][10][2:14]
for phase in (0, 1):
    up = 'u' if phase == 0 else 'n'
    print('phase', phase, ''.join('#' if c == up else '.' for c in fence), '(# = raised, . = walkable)')


## 2026-10-20: Animated GoDGuilD logo on the title screen

**What:** the title screen gains the 2026 animated GoDGuilD logo, small (52x42 px) in the bottom-right corner. `src/ui/GgLogo.gd` (a `TextureRect`) plays three spritesheets in `assets/ui/gg_logo/` (224x182 frames, 15 fps): intro, idle loop, and every 25-55 s a 1 s glitch-out followed by the intro again. Both are copies from the logo package (`H:/OneDrive/Design/GoDGuilD/2026anim_lgo`: `src/export_godot.py` renders the sheets, `player/godot/GgLogo.gd` is the player); regenerate there and copy over.

**Why linear filtering here:** the project renders 384x216 with the nearest filter, which is right for the pixel art but turns a 224 px chrome logo shrunk to 52 px into noise. The logo node sets `TEXTURE_FILTER_LINEAR` for itself only; with `canvas_items` stretch it is drawn at window resolution, so it stays smooth.

**Why not as pixel art:** the logo is painted chrome with soft glow; a hand-made pixel version would be a separate art job. The corner placement keeps it out of the menu and the game's own title art.


## 2026-10-20 — Phase 12: Silva (deep water, the flippers, swimming)

**Water gets its own physics layer.** Until now deep water was simply a solid tile on the world layer, which
everything collides with. The flippers need *only Joe* to stop colliding with it, so a water cell's collision moved
to a third TileSet physics layer (10, `LAYER_WATER`), chosen by a new `water` flag in the atlas manifest (the cell
stays `solid` there, so nothing that reads the manifest's solidity changes). Then each body decides: Joe's mask
comes from `Player.solid_mask()` (re-read every frame, so finding the flippers takes effect at once); enemies and
push blocks always add the water layer. Projectiles and thrown objects mask the world layer only, so they now fly
over water the way they already flew over pits. That is a behaviour change, accepted and documented.

Rejected: a swim *ability* that turned water cells into floor at runtime (like the ice rod's crust). That would
have needed every enemy to keep treating those cells as walls, and the tiles are shared by the whole area.

**The mirror needed the same rule.** `World._solid_at` decides whether a Moon Mirror warp may land, by asking
whether the tile has a collision polygon on physics layer 0. After the move, water had none there, so without a
fix a mirror warp could have dropped a flipper-less Joe in a lake. It now checks the `water` flag explicitly.

**Swimming is a state, not a speed modifier.** In ALttP, swimming takes away the sword and items. A `swim` state
that idle, walk, dash and carry all hand over to (via `_check_water`, next to `_check_pit`) gets that for free:
the state simply never reads the attack or item buttons. A hit in the water goes through `hurt` and back to
`swim`; a flag stops the splash from replaying.

**The lint treats deep water as a step** (as it treats grapple and ice hops: every item is assumed held). Switching
the step off makes the far bank, the islet chest and the boss island unreachable, which shows it is a real gate.
Like the molten time, the Grove hall's pool and the Thorn Woods' pegs are proven in the game.

**Gotchas:** running the unchanged `gen_joe.py` redraws the sword strips with small pixel differences (old drift,
probably a Pillow update), so only the new swim strips were kept. The Dark World bosses' "angry" threshold read
`max_hp`, the stat before the World doubles their health, so they sped up at a quarter of their health instead of
half.

The cell below lists, from the generated areas, which cells of the Silva Tangle only the flippers reach:


In [ ]:
import json, os
from collections import deque

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'technical_memory' else os.getcwd()
a = json.load(open(os.path.join(ROOT, 'src', 'world', 'areas', 'dw_silva_tangle.json'), encoding='utf-8'))
rows = a['rows']
WALK = set('.,*#g')                       # plain ground of the Tangle (actors stand on ground too)


def reach(swim):
    start = (len(rows[0]) - 1, 13)        # the east opening from the Thorn Woods
    seen, q = {start}, deque([start])
    while q:
        x, y = q.popleft()
        for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nx, ny = x + dx, y + dy
            if 0 <= nx < len(rows[0]) and 0 <= ny < len(rows) and (nx, ny) not in seen:
                g = rows[ny][nx]
                if g in WALK or g in 'jhBA' or (swim and g == '~'):
                    seen.add((nx, ny))
                    q.append((nx, ny))
    return seen


walk, swim = reach(False), reach(True)
far = sorted((x, y) for x, y in swim - walk if rows[y][x] != '~')
print('cells reachable on foot:', len(walk), '- with the flippers:', len(swim))
print('far-bank cells only the flippers reach:', len(far))
print('the piece of heart (4, 14) needs them:', (4, 14) in far)


## 2026-10-20 — Phase 13: Caver's tower (the final boss and the ending)

**One gate class, two gates.** The tower's crystal gate behaves exactly like the keep's seal gate (talkable, sockets,
a flag, opens for good), so it is `SealGate` with `crystal = true` rather than a subclass: the textures and socket
positions are preloaded constants, and a subclass would have had to duplicate them or override `_ready` wholesale.
The only real difference - which sockets light - is the optional `which` array of `refresh()` (crystals are found in
any order; seals were counted).

**Endings became data.** `play_ending` had the Act 1 keys hard-coded. A second ending turned them into
`World.ENDINGS` (`act` -> villain's words + card) and the save spot into `act2_spawn` / `final_spawn`. The final
ending saves Joe in front of the tower and returns to the title, like Act 1: there are no credits until Phase 15.

**The fight: Agahnim and Ganon in one.** Caver keeps the shade's tennis (only a batted orb touches him) but a
batted orb no longer ends the exchange: it knocks him down for `DAZE_TIME`, and only then do ordinary hits land.
That makes the sword matter in the last fight. Volleys of bolts (every third cast, every second when angry) are
blockable by the shield, so they reward the ALttP habit of facing the danger. His death overrides `_die` - no poof,
a tween that shrinks the sprite toward his feet (the sprite's origin is the feet, so scaling `y` reads as sinking) -
and still emits `defeated`, which is all the World needs.

**The lint's blind spot, again.** Removing the Umbra floor's hook posts left the lint silent: the room's north
doorway scrolls into the Ignis floor, and every scroll opening counts as an entry, so the far side looks reachable
from above. The same holds for the lava band. Making the lint follow dungeon progress (entries only from the way
Joe came) would be a real feature; for now the tower's gates are crossed in `test_tower.gd`, as Silva's were.
The pool *is* caught (the big key's islet has no other way in).

**Gotchas found by the new suite and the captures:**
- Leaving a room within 6 s of freezing lava printed "Lambda capture was freed" errors: the thaw timer's lambda
  captured the room's `TileMapLayer`. A lambda that can outlive its node must hold a `WeakRef`, and `weakref()`
  (like `get_ref()`) returns Variant, so `:=` fails under warnings-as-errors - declare the types.
- The map drew every screen at a fixed size; the Dark World had been running off the panel since Umbra (seven
  screens tall). Boxes now shrink per axis, and names wrap - a schematic map is fine for ALttP's style.
- Bolts in the air outlived Caver and could hit Joe as the ending started: a boss's shots now die with it.

The cell below rebuilds, from the generated rooms, what each tower floor needs: for each gate glyph, whether the
room's goal (its north doorway, or the big key's chest) is reachable from where Joe enters with that terrain
treated as a wall, and with it treated as floor (the item):


In [ ]:
import json, os
from collections import deque

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'technical_memory' else os.getcwd()
FLOOR = set('=;@su n'.replace(' ', ''))       # dungeon floor, switch, stairs, tick blocks (half are always down)
# room -> (gate glyph and the item that crosses it, entry cell, goal cells)
FLOORS = {
    'd8_umbra': ('o', 'grapple', (11, 11), [(11, 2), (12, 2)]),
    'd8_ignis': ('!', 'ice rod', (11, 11), [(11, 2), (12, 2)]),
    'd8_tempus': ('p', 'hammer', (11, 11), [(11, 2), (12, 2)]),
    'd8_silva': ('~', 'flippers', (21, 6), [(10, 7)]),
}


def reach(rows, start, gate, crossed):
    ok = FLOOR | ({gate} if crossed else set())
    seen, q = {start}, deque([start])
    while q:
        x, y = q.popleft()
        for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nx, ny = x + dx, y + dy
            if 0 <= ny < len(rows) and 0 <= nx < len(rows[0]) and (nx, ny) not in seen:
                g = rows[ny][nx]
                if g in ok or g.isalpha() and g not in 'pou':   # enemies stand on floor
                    seen.add((nx, ny))
                    q.append((nx, ny))
    return seen


for room, (gate, item, start, goals) in FLOORS.items():
    rows = json.load(open(os.path.join(ROOT, 'src', 'world', 'areas', room + '.json'), encoding='utf-8'))['rows']
    without = any(g in reach(rows, start, gate, False) for g in goals)
    with_it = any(g in reach(rows, start, gate, True) for g in goals)
    print('%-10s gate %r (%-8s): goal reachable without it: %-5s with it: %s' % (room, gate, item, without, with_it))


## 2026-10-21 — Phase 14: polish, audio, localization

**Translation is a draw-time lookup, ported from Chapel.** `PixelText` keeps the English `source` and draws
`Loc.t(source)`, so code keeps writing English everywhere and only the tables change. Two rules make it work:
translate *before* measuring (caption plates, map labels) and translate the *parts* of composed labels (the item menu
translates the name, then appends the count). Dialogue is different: its pages are long and wrapped, so a per-line
lookup would never match. `DialogueData` lays a whole Spanish table over the English one - text fields only, so a
translation can never change a flag, a gift or an item - and the dialogue box switches `translate` off for its lines.
Text captured by a `%s` placeholder is translated too, which turned "PRESS %s" + "CIRCLE" into "PULSA CÍRCULO"
without a key per button.

**Live language switching.** Every PixelText joins a group; `Loc.refresh` re-sets each one's source, so the pause
menu behind the options screen changes language at once. Rejected: rebuilding every screen on a signal (each menu
would need its own rebuild code).

**Fonts: accents need a free row.** The 5x7 capitals use 7 of the cell's 8 rows. Chapel's approach - mark on row 0,
letter one row shorter - left the mark touching the letter, and at 1x it simply vanished (the captures read MUSICA,
ESPANOL). A free row between mark and letter fixes legibility at the cost of a one-pixel baseline drop for accented
capitals; the alternative (two-row-shorter "small caps" accented letters) looked worse.

**Music is composed, not hand-written.** Ten tracks from specs (tempo, key, mode, one chord per bar, arrangement): a
seeded generator writes a four-bar motif (rhythms from a pool, scale-step contour, chord tones on beats 1 and 3),
answers it over the next chords keeping the contour, contrasts it in the B section and ends on the tonic. Every note
is overlap-added into a circular buffer, so release tails wrap into bar 1 and the loops are seamless (the generator
checks the seam). Kept as 22 kHz WAV (13 MB) for sample-exact loops, like Chapel; Ogg would be a tenth of the size.

**Settings live in `Audio`** (as in Chapel), and a `-s` script never reads the player's file: otherwise a player
who chose Spanish would make every suite fail. Detection is simple: the game's main loop has no script, a test's has.

**Gamepad.** SNES layout: the right face button confirms (SNES A), the bottom one goes back (SNES B) - `ui_cancel`
had no joypad event at all before. Hints name the pad's own confirm button by family (Xbox B, Nintendo A,
PlayStation circle), from `Input.get_joy_name`. Only the four stick axes count as "using the pad": a capture showed
a resting trigger reporting full deflection.

The cell below loads the generated music manifest and the Spanish tables and summarises them:


In [ ]:
import json, os

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'technical_memory' else os.getcwd()
music = json.load(open(os.path.join(ROOT, 'assets', 'audio', 'music', 'music_manifest.json'), encoding='utf-8'))
total = sum(t['duration_s'] for t in music.values())
for name, t in music.items():
    print('%-11s %5.1f s  %3d bpm  %2d bars  rms %6.2f dB' % (name, t['duration_s'], t['bpm'], t['bars'], t['rms_db']))
print('music: %d tracks, %.0f s of loops' % (len(music), total))
ui = {k: v for k, v in json.load(open(os.path.join(ROOT, 'localization', 'es.json'), encoding='utf-8')).items() if not k.startswith('_')}
dia = {k: v for k, v in json.load(open(os.path.join(ROOT, 'localization', 'dialogue_es.json'), encoding='utf-8')).items() if not k.startswith('_')}
pages = sum(len(v.get('pages', [])) + len(v.get('once', [])) + len(v.get('gift_pages', [])) for v in dia.values())
accented = sorted({c for v in ui.values() for c in v if ord(c) > 127} | {c for v in dia.values() for p in v.get('pages', []) for c in p if ord(c) > 127})
print('Spanish: %d UI strings, %d dialogue entries (%d pages); letters beyond ASCII: %s' % (len(ui), len(dia), pages, ''.join(accented)))
